## Silver Layer Transformations

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Data Loading

In [0]:
df_cal = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Calendar")

In [0]:
df_cal.display()

In [0]:
df_cus = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Customers")

In [0]:
df_cat = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Product_Categories")

In [0]:
df_subcat = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Product_Subcategories")

In [0]:
df_pro = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Products")

In [0]:
df_ret = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Returns")

In [0]:
df_sales = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Sales*")

In [0]:
df_ter = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Territories")

## TRANSFORMATIONS

#### Calender

In [0]:
df_cal.display()

In [0]:
df_cal = df_cal.withColumn('Month', month(col('Date')))\
               .withColumn('Year', year(col('Date')))
df_cal.display()

In [0]:
df_cal.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Calendar")\
    .save()


#### Customers

In [0]:
df_cus.display()

In [0]:
df_cus = df_cus.withColumn('FullName', concat_ws(' ', col("Prefix"), col("FirstName"), col("LastName")))
df_cus.display()

In [0]:
df_cus.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Customers")\
    .save()

#### Subcategories

In [0]:
df_cat.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Product_Categories")\
    .save()

In [0]:
df_subcat.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Product_Subcategories")\
    .save()

#### Products

In [0]:
df_pro.display()

In [0]:
#We want all the alphabets before ProductSKU before -
#In product name fetch first word before space for every row in that column

#we are using split function
#We are also indexing

df_pro = df_pro.withColumn('ProductSKU', split(col("ProductSKU"), '-')[0])\
               .withColumn('ProductName', split(col("ProductName"), ' ')[0])
df_pro.display()


In [0]:
df_pro.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Products")\
    .save()

#### Returns

In [0]:
df_ret.display()

In [0]:
df_ret.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Returns")\
    .save()

In [0]:
df_ter.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Territories")\
    .save()

#### Sales

In [0]:
df_sales.display()

In [0]:
#REQUIREMENTS

#converting date to timestamp - stockDate
#Replacing an alpabet with anothet say replacing s with t
#mathematical operations - multiplying orderlineitem with orderquantity
#Performing aggregations

df_sales = df_sales.withColumn('StockDate', to_timestamp(col("StockDate")))
                   

In [0]:
df_sales = df_sales.withColumn('OrderNumber', regexp_replace(col('OrderNumber'), 'S', 'T'))

In [0]:
df_sales = df_sales.withColumn('multiply', col("OrderLineItem")* col("OrderQuantity"))

In [0]:
df_sales.display()

In [0]:
df_sales.write.format("parquet")\
    .mode("append")\
    .option("path","abfss://silver@awstoragedatalakenit.dfs.core.windows.net/AdventureWorks_Sales")\
    .save()

#### Sales Analysis

In [0]:
#performing group aggregate

#finding how many orders we recieved everyday

#So we have to groupby orderdate, count

df_sales.groupBy('OrderDate').agg(count('OrderNumber').alias('totalOrders')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_cat.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_ter.display()

Databricks visualization. Run in Databricks to view.